In [1]:
import json
with open('../data/example-fmt.json') as f:
  data = json.load(f)

In [ ]:
import base64
from shapely import wkb
route_geom = wkb.loads(base64.b64decode(data['routes'][0]['geom_b64']))

def parse_routes(route_data):
  """Parse route geometries"""
  route_geoms = []
  for route_obj in route_data:
    geom = wkb.loads(base64.b64decode(route_obj['geom_b64']))
    tmp = route_obj.copy()
    del tmp['geom_b64']
    tmp['geom'] = geom
    route_geoms.append(tmp)
  return route_geoms

def parse_map_issues(map_issues):
  """Convert map issues into protobufs (?)"""
  output = []
  for mi in map_issues:
    geom = wkb.loads(bytes.fromhex(mi['reported_location_hexwkb']))
    tmp = mi.copy()
    del tmp['reported_location_hexwkb']
    tmp['reported_location'] = dict(lng=geom.xy[0][0], lat=geom.xy[1][0])
    output.append(tmp)
  return output

parse_routes(data['routes'])
parse_map_issues(data['map_issues'])

[{'id': 'bcb09d44-4221-4263-bf84-e9d53add0787',
  'issue_type': 'Homeless',
  'created_at': 1759605544.808862,
  'notes': '',
  'reported_location': {'lng': -118.30044872749431, 'lat': 34.03718777534139}},
 {'id': '95f2f1ae-3b39-470f-a612-e0317b0f26b8',
  'issue_type': 'Homeless',
  'created_at': 1759606058.700801,
  'notes': '',
  'reported_location': {'lng': -118.30013222865918, 'lat': 34.03730339471919}},
 {'id': '200e0d06-ebeb-4424-b0c3-243a481f235b',
  'issue_type': 'Homeless',
  'created_at': 1759606826.198505,
  'notes': '',
  'reported_location': {'lng': -118.29839424287135,
   'lat': 34.036409317615345}}]

In [18]:
fp = '/Users/bradsquicciarini/Downloads/coco-openai_20241206/C10899__1731191615003833265-1731191679999554007.mcap'

In [19]:
from mcap.reader import make_reader

In [22]:
from mcap_protobuf.decoder import DecoderFactory
with open(fp, 'rb') as f:
  reader = make_reader(f, decoder_factories=[DecoderFactory()])
  for s, c, dm, m in reader.iter_decoded_messages(topics=['/route/issue_report']):
    break

In [ ]:
from google.protobuf.descriptor_pb2 import FileDescriptorProto
from google.protobuf.text_format import MessageToString

fd = FileDescriptorProto()
fd.ParseFromString(s.data)

fd0 = FileDescriptorProto()
fd0.ParseFromString(fd.name.encode())

print(MessageToString(fd0))

TypeError: expected bytes, str found

In [36]:
fd.name.encode()

b'\n\x15proto/MapReport.proto\x12\x04coco"$\n\x08GeoPoint\x12\x0b\n\x03lat\x18\x01 \x01(\x02\x12\x0b\n\x03lng\x18\x02 \x01(\x02"d\n\tMapReport\x12\x1d\n\x04type\x18\x01 \x01(\x0e2\x0f.coco.IssueType\x12\r\n\x05notes\x18\x02 \x01(\t\x12)\n\x11reported_location\x18\x03 \x01(\x0b2\x0e.coco.GeoPoint*9\n\tIssueType\x12\x0f\n\x0bISSUE_OTHER\x10\x00\x12\x1b\n\x17ISSUE_ROUTE_OBSTRUCTION\x10\x01b\x06proto3'